# 🚢 泰坦尼克号生存预测案例

**Titanic - Machine Learning from Disaster**

本 Notebook 将完整演示如何利用机器学习预测泰坦尼克号乘客的生存概率。

## 📋 目录
1. [项目概述](#1-项目概述)
2. [数据加载与探索](#2-数据加载与探索)
3. [数据清洗](#3-数据清洗)
4. [特征工程](#4-特征工程)
5. [模型训练与评估](#5-模型训练与评估)
6. [超参数调优](#6-超参数调优)
7. [集成学习](#7-集成学习)
8. [生成提交文件](#8-生成提交文件)

## 🔰 背景介绍
1912年4月15日，泰坦尼克号在撞上冰山后沉没，2224名乘客和机组人员中约1500人遇难。
本案例目标：**基于乘客特征（性别、舱位、年龄等）预测其是否生还**。

## 📊 数据字典
| 特征 | 描述 | 类型 |
|------|------|------|
| PassengerId | 乘客ID | 数值 |
| Survived | 是否生还 (0/1) | 目标变量 |
| Pclass | 舱位等级 (1/2/3) | 有序类别 |
| Name | 姓名 | 文本 |
| Sex | 性别 | 类别 |
| Age | 年龄 | 数值 |
| SibSp | 同船兄弟姐妹/配偶数 | 数值 |
| Parch | 同船父母/子女数 | 数值 |
| Ticket | 船票号 | 文本 |
| Fare | 票价 | 数值 |
| Cabin | 客舱号 | 文本 |
| Embarked | 登船港口 (C/Q/S) | 类别 |

## 1️⃣ 项目概述

In [ ]:
# 导入必要的库
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# 项目模块
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print('库导入完成 ✓')
print(f'Python版本: {sys.version.split()[0]}')
print(f'NumPy版本: {np.__version__}')
print(f'Pandas版本: {pd.__version__}')
print(f'Scikit-learn版本: {import sklearn; print(sklearn.__version__)}')

## 2️⃣ 数据加载与探索

In [ ]:
# 加载数据
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'kaggle', 'titanic', 'data')

try:
    train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    print(f'✅ 数据加载成功！')
    print(f'   训练集: {train_df.shape[0]} 行 × {train_df.shape[1]} 列')
    print(f'   测试集: {test_df.shape[0]} 行 × {test_df.shape[1]} 列')
except FileNotFoundError:
    print('❌ 数据文件不存在！请先下载数据集：')
    print('   Kaggle: https://www.kaggle.com/c/titanic/data')
    print(f'   放置路径: {DATA_DIR}')

In [ ]:
# 数据基本信息
print('=' * 60)
print('【训练集概览】')
print('=' * 60)
print(train_df.info())
print()
print('【数据类型】')
print(train_df.dtypes)
print()
print('【前10行数据】')
train_df.head(10)

In [ ]:
# 缺失值分析
print('=' * 60)
print('【缺失值统计】')
print('=' * 60)

missing_data = pd.DataFrame({
    '训练集缺失数': train_df.isnull().sum(),
    '训练集缺失率%': (train_df.isnull().sum() / len(train_df) * 100).round(2),
    '测试集缺失数': test_df.isnull().sum(),
    '测试集缺失率%': (test_df.isnull().sum() / len(test_df) * 100).round(2),
})
missing_data = missing_data[missing_data['训练集缺失数'] > 0]
print(missing_data)

print(f"""
📌 缺失值情况：
   - Cabin 缺失严重 (~77%)：大部分乘客没有指定客舱
   - Age 缺失 (~20%)：可用其他特征推断
   - Embarked 缺失极少 (仅2条)：可用众数填充
""")

In [ ]:
# 数值列统计
print('【数值特征统计描述】')
train_df.describe()

In [ ]:
# 🔍 单变量探索 - 生存率
print('=' * 60)
print('【生存率分析】')
print('=' * 60)

survival_rate = train_df['Survived'].mean()
print(f'\n总体生存率: {survival_rate:.2%}')

# 按性别分析
print('\n📊 性别 × 生存率:')
sex_surv = train_df.groupby('Sex')['Survived'].agg(['sum', 'count', 'mean'])
sex_surv.columns = ['存活数', '总数', '生存率']
print(sex_surv)

# 按舱位分析
print('\n📊 舱位等级 × 生存率:')
pclass_surv = train_df.groupby('Pclass')['Survived'].agg(['sum', 'count', 'mean'])
pclass_surv.columns = ['存活数', '总数', '生存率']
print(pclass_surv)

# 按登船港口分析
print('\n📊 登船港口 × 生存率:')
embarked_surv = train_df.groupby('Embarked')['Survived'].agg(['sum', 'count', 'mean'])
embarked_surv.columns = ['存活数', '总数', '生存率']
print(embarked_surv)

print(f"""
💡 关键发现：
   - 女性生存率 (~74%) 远高于男性 (~19%)
   - 头等舱生存率 (~63%) 远高于三等舱 (~24%)
   - 法国港口(C)生存率最高 (~55%)
""")

In [ ]:
# 📈 可视化 - 多维度分布图
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. 生存分布
axes[0, 0].hist(train_df['Survived'], bins=2, edgecolor='black', color=['#2ca02c', '#d62728'])
axes[0, 0].set_title('Survived Distribution', fontsize=12)
axes[0, 0].set_xlabel('Survived (0=No, 1=Yes)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(['Not Survived', 'Survived'])

# 2. 性别 × 生存率
male_rate = train_df[train_df['Sex'] == 'male']['Survived'].mean()
female_rate = train_df[train_df['Sex'] == 'female']['Survived'].mean()
axes[0, 1].bar(['Male', 'Female'], [male_rate, female_rate], 
                color=['#1f77b4', '#ff7f0e'], edgecolor='black')
axes[0, 1].set_title('Survival Rate by Sex', fontsize=12)
axes[0, 1].set_ylabel('Survival Rate')
axes[0, 1].set_ylim(0, 1)
for i, v in enumerate([male_rate, female_rate]):
    axes[0, 1].text(i, v + 0.02, f'{v:.1%}', ha='center')

# 3. 舱位 × 生存率
pclass_groups = train_df.groupby('Pclass')['Survived'].mean()
bar_colors = ['#2ca02c', '#ff7f0e', '#d62728']
axes[0, 2].bar(['1st', '2nd', '3rd'], pclass_groups.values, 
                color=bar_colors, edgecolor='black')
axes[0, 2].set_title('Survival Rate by Pclass', fontsize=12)
axes[0, 2].set_ylabel('Survival Rate')
axes[0, 2].set_ylim(0, 1)
for i, v in enumerate(pclass_groups.values):
    axes[0, 2].text(i, v + 0.02, f'{v:.1%}', ha='center')

# 4. 年龄分布
age_survived = train_df[train_df['Survived'] == 1]['Age'].dropna()
age_not_survived = train_df[train_df['Survived'] == 0]['Age'].dropna()
axes[1, 0].hist(age_survived, bins=30, alpha=0.6, label='Survived', color='#2ca02c')
axes[1, 0].hist(age_not_survived, bins=30, alpha=0.6, label='Not Survived', color='#d62728')
axes[1, 0].set_title('Age Distribution by Survival', fontsize=12)
axes[1, 0].set_xlabel('Age')
axes[1, 0].legend()

# 5. 票价分布
axes[1, 1].hist(train_df['Fare'].dropna(), bins=50, edgecolor='black', color='#9467bd')
axes[1, 1].set_title('Fare Distribution', fontsize=12)
axes[1, 1].set_xlabel('Fare')
axes[1, 1].set_ylabel('Count')

# 6. 登船港口分布
embarked_counts = train_df['Embarked'].value_counts().sort_index()
axes[1, 2].pie(embarked_counts.values, labels=embarked_counts.index, 
               autopct='%1.1f%%', colors=['#2ca02c', '#ff7f0e', '#1f77b4'])
axes[1, 2].set_title('Embarked Distribution', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 🔥 相关性热力图
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
            square=True, fmt='.2f', ax=ax, 
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap - Titanic Features', fontsize=14)
plt.tight_layout()
plt.show()

print("""
💡 相关性洞察：
   - Survived 与 Sex (0.54) 强相关：女性更易生还
   - Survived 与 Pclass (-0.34) 负相关：舱位越高越难生还
   - Survived 与 Fare (0.26) 正相关：票价越高更易生还
   - Age 与 Survived 相关性较弱 (-0.07)
""")

## 3️⃣ 数据清洗

处理策略：
- **Age**：使用 Sex × Pclass 分组的中位数填充
- **Embarked**：使用众数填充
- **Fare**：使用 Pclass × Embarked 分组的中位数填充
- **Cabin**：提取甲板信息，标记有无舱位

In [ ]:
# 数据清洗流程
def clean_data(train_df, test_df):
    """完整的数据清洗流程"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # 1. 处理 Age 缺失值
    for df in [train_df, test_df]:
        age_median = df.groupby(['Sex', 'Pclass'])['Age'].transform('median')
        df['Age'] = df['Age'].fillna(age_median)
        df['Age'] = df['Age'].fillna(df['Age'].median())
    
    # 2. 处理 Embarked 缺失值
    for df in [train_df, test_df]:
        df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # 3. 处理 Fare 缺失值
    for df in [train_df, test_df]:
        fare_median = df.groupby(['Pclass', 'Embarked'])['Fare'].transform('median')
        df['Fare'] = df['Fare'].fillna(fare_median)
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # 4. 处理 Cabin 缺失值
    for df in [train_df, test_df]:
        df['Cabin_deck'] = df['Cabin'].apply(
            lambda x: str(x)[0] if pd.notna(x) and str(x).strip() != '' else 'Unknown'
        )
        df['Has_Cabin'] = df['Cabin'].notna().astype(int)
    
    return train_df, test_df

# 执行清洗
train_df, test_df = clean_data(train_df, test_df)

# 验证缺失值
print('【清洗后缺失值统计】')
for col in train_df.columns:
    t_miss = train_df[col].isna().sum()
    te_miss = test_df[col].isna().sum()
    if t_miss > 0 or te_miss > 0:
        print(f'  {col}: 训练集{t_miss}, 测试集{te_miss}')
    else:
        print(f'  {col}: ✓ 完整')

## 4️⃣ 特征工程

特征工程是机器学习中最重要的环节之一。我们将创建以下新特征：

1. **Name → Title**：提取称谓（Mr/Mrs/Miss/Master）
2. **Age → Age_group**：年龄分组（儿童/青少年/成人/老年）
3. **Ticket → Ticket_prefix**：船票前缀
4. **SibSp + Parch → Family_size**：家庭规模
5. **Fare → Fare_log**：对数变换
6. **Sex → 编码**：标签编码

In [ ]:
def engineer_features(train_df, test_df):
    """完整的特征工程流程"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # ===== 1. 从姓名提取特征 =====
    for df in [train_df, test_df]:
        # 提取称谓
        df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
        
        # 合并稀有称谓
        title_mapping = {
            'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master',
            'Mme': 'Mrs', 'Mlle': 'Miss', 'Ms': 'Miss',
            'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare',
            'Major': 'Rare', 'Don': 'Rare', 'Dona': 'Rare',
            'Lady': 'Rare', 'Sir': 'Rare', 'the Countess': 'Rare',
            'Jonkheer': 'Rare', 'Capt': 'Rare',
        }
        df['Title'] = df['Title'].map(title_mapping).fillna('Rare')
    
    # ===== 2. 年龄特征 =====
    for df in [train_df, test_df]:
        # 年龄分组
        bins = [0, 12, 18, 25, 35, 50, 65, 100]
        labels = ['0-12', '13-18', '19-25', '26-35', '36-50', '51-65', '66+']
        df['Age_group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)
        
        # 是否为儿童
        df['Is_Child'] = (df['Age'] < 18).astype(int)
        
        # 年龄 × 舱位交互
        df['Age_Pclass'] = df['Age'] * df['Pclass']
    
    # ===== 3. 船票特征 =====
    for df in [train_df, test_df]:
        # 票号前缀
        df['Ticket_prefix'] = df['Ticket'].str.extract(r'^([A-Za-z])')
        df['Ticket_prefix'] = df['Ticket_prefix'].fillna('NONE')
        
        # 是否纯数字票号
        df['Ticket_is_numeric'] = df['Ticket'].str.match(r'^\d+$').astype(int)
    
    # ===== 4. 票价特征 =====
    for df in [train_df, test_df]:
        # 票价分组
        bins = [0, 7.91, 14.45, 31, 1000]
        labels = ['Low', 'Mid_Low', 'Mid', 'High']
        df['Fare_group'] = pd.cut(df['Fare'], bins=bins, labels=labels, right=False)
        
        # 对数票价
        df['Fare_log'] = np.log1p(df['Fare'])
        
        # 人均票价
        df['Fare_per_person'] = df['Fare'] / (df['SibSp'] + df['Parch'] + 1)
    
    # ===== 5. 家庭特征 =====
    for df in [train_df, test_df]:
        # 家庭规模
        df['Family_size'] = df['SibSp'] + df['Parch'] + 1
        
        # 是否单独旅行
        df['Is_Alone'] = (df['Family_size'] == 1).astype(int)
        
        # 家庭类型
        df['Family_type'] = df['Family_size'].apply(
            lambda x: 'Solo' if x == 1 else ('Small' if x <= 4 else 'Large')
        )
    
    # ===== 6. 合并训练集和测试集进行编码 =====
    combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    
    # 类别特征 One-Hot 编码
    categorical_cols = ['Embarked', 'Title', 'Family_type', 'Ticket_prefix',
                        'Cabin_deck', 'Age_group', 'Fare_group']
    for col in categorical_cols:
        if col in combined.columns:
            dummies = pd.get_dummies(combined[col], prefix=col)
            combined = pd.concat([combined, dummies], axis=1)
            combined = combined.drop(columns=[col])
    
    # 二元特征标签编码
    binary_cols = ['Sex']
    for col in binary_cols:
        if col in combined.columns:
            le = LabelEncoder()
            combined[f'{col}_encoded'] = le.fit_transform(combined[col])
            combined = combined.drop(columns=[col])
    
    # 分割回训练集和测试集
    train_df = combined.iloc[:len(train_df)].copy()
    test_df = combined.iloc[len(train_df):].copy()
    
    # 移除不需要的列
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    train_df = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns], errors='ignore')
    test_df = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns], errors='ignore')
    
    # 去除重复列
    train_df = train_df.loc[:, ~train_df.columns.duplicated()]
    test_df = test_df.loc[:, ~test_df.columns.duplicated()]
    
    return train_df, test_df

# 执行特征工程
train_fe, test_fe = engineer_features(train_df, test_df)

print(f'【特征工程完成】')
print(f'  训练集: {train_fe.shape[0]} 行 × {train_fe.shape[1]} 列')
print(f'  测试集: {test_fe.shape[0]} 行 × {test_fe.shape[1]} 列')

# 查看所有特征
feature_cols = [c for c in train_fe.columns if c != 'Survived']
print(f'\n【特征列表】({len(feature_cols)}个)')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

## 5️⃣ 模型训练与评估

In [ ]:
# 准备训练数据
X = train_fe.drop('Survived', axis=1)
y = train_fe['Survived']
X_test = test_fe[X.columns]

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'数据划分:')
print(f'  训练集: {X_train.shape[0]} 样本')
print(f'  验证集: {X_val.shape[0]} 样本')
print(f'  测试集: {X_test.shape[0]} 样本')
print(f'  特征数: {X.shape[1]}')

In [ ]:
# 初始化多种模型
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=42),
}

# 评估所有模型
results = []

for name, model in models.items():
    print(f'\n训练: {name} ...')
    
    # 训练
    model.fit(X_train, y_train)
    
    # 预测
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # 计算指标
    metrics = {
        'Model': name,
        'Train_Accuracy': accuracy_score(y_train, y_train_pred),
        'Val_Accuracy': accuracy_score(y_val, y_val_pred),
        'Val_Precision': precision_score(y_val, y_val_pred),
        'Val_Recall': recall_score(y_val, y_val_pred),
        'Val_F1': f1_score(y_val, y_val_pred),
    }
    
    # AUC
    try:
        y_prob = model.predict_proba(X_val)[:, 1]
        metrics['Val_AUC'] = roc_auc_score(y_val, y_prob)
    except:
        metrics['Val_AUC'] = None
    
    results.append(metrics)
    print(f'  训练准确率: {metrics["Train_Accuracy"]:.4f}')
    print(f'  验证准确率: {metrics["Val_Accuracy"]:.4f}')
    print(f'  验证F1: {metrics["Val_F1"]:.4f}')

# 汇总结果
results_df = pd.DataFrame(results).sort_values('Val_Accuracy', ascending=False)
print(f'\n{"="*60}')
print('【模型性能排名】')
print(f'{"="*60}')
print(results_df[['Model', 'Train_Accuracy', 'Val_Accuracy', 'Val_F1', 'Val_AUC']].to_string(index=False))

In [ ]:
# 📊 交叉验证（5折）
print('=' * 60)
print('【5折交叉验证结果】')
print('=' * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'CV_Mean': scores.mean(),
        'CV_Std': scores.std(),
    })
    print(f'  {name}: {scores.mean():.4f} ± {scores.std():.4f}')

cv_df = pd.DataFrame(cv_results).sort_values('CV_Mean', ascending=False)
print(f'\n  最佳CV模型: {cv_df.iloc[0]["Model"]} ({cv_df.iloc[0]["CV_Mean"]:.4f})')

In [ ]:
# 🔍 ROC曲线对比
fig, ax = plt.subplots(figsize=(10, 8))

for name, model in models.items():
    try:
        y_prob = model.predict_proba(X_val)[:, 1]
        fpr, tpr, _ = roc_curve(y_val, y_prob)
        auc = roc_auc_score(y_val, y_prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    except:
        pass

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves Comparison', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 🔥 混淆矩阵 - 最佳模型
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

print(f'【最佳模型】{best_model_name}')

y_val_pred = best_model.predict(X_val)
cm = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
ax.set_title(f'Confusion Matrix - {best_model_name}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

print(f'\n【分类报告】')
print(classification_report(y_val, y_val_pred, target_names=['Not Survived', 'Survived']))

In [ ]:
# 📊 特征重要性
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:15]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(15), importances[indices][::-1], color='#2ca02c')
    ax.set_yticks(range(15))
    ax.set_yticklabels([X.columns[i] for i in indices[::-1]])
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top 15 Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
    
    print('\n【Top 10 重要特征】')
    for i in range(10):
        idx = indices[i]
        print(f'  {i+1:2d}. {X.columns[idx]:<30} {importances[idx]:.4f}')

## 6️⃣ 超参数调优

In [ ]:
# 对最佳模型进行超参数调优
print(f'超参数调优: {best_model_name}')

if best_model_name in ['RandomForest']:
    param_dist = {
        'n_estimators': [100, 200, 500],
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
    }
elif best_model_name in ['GradientBoosting']:
    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'min_samples_split': [2, 5, 10],
    }
elif best_model_name in ['LogisticRegression']:
    param_dist = {
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': [500, 1000, 2000],
    }
else:
    param_dist = {'n_estimators': [100, 200]}

# 使用 RandomizedSearchCV
search = RandomizedSearchCV(
    best_model.__class__(**best_model.get_params()),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
)

search.fit(X, y)

tuned_model = search.best_estimator_
tuned_score = search.best_score_

print(f'\n  最佳参数: {search.best_params_}')
print(f'  最佳CV分数: {tuned_score:.4f}')

# 评估调优后的模型
tuned_val_pred = tuned_model.predict(X_val)
tuned_val_acc = accuracy_score(y_val, tuned_val_pred)
print(f'  调优后验证集准确率: {tuned_val_acc:.4f}')
print(f'  提升: {tuned_val_acc - results_df.iloc[0]["Val_Accuracy"]:.4f}')

## 7️⃣ 集成学习

使用投票法（Voting）将多个强模型组合起来，通常可以获得更稳定的结果。

In [ ]:
# 创建投票集成模型
print('创建投票集成模型 ...')

# 选择前3个模型作为基础模型
top3_models = [
    (results_df.iloc[0]['Model'], models[results_df.iloc[0]['Model']]),
    (results_df.iloc[1]['Model'], models[results_df.iloc[1]['Model']]),
    (results_df.iloc[2]['Model'], models[results_df.iloc[2]['Model']]),
]

# Soft Voting (基于概率)
ensemble_soft = VotingClassifier(
    estimators=top3_models,
    voting='soft',
    n_jobs=-1,
)

# Hard Voting (基于类别)
ensemble_hard = VotingClassifier(
    estimators=top3_models,
    voting='hard',
    n_jobs=-1,
)

# 训练和评估
for name, ensemble in [('Soft Voting', ensemble_soft), ('Hard Voting', ensemble_hard)]:
    ensemble.fit(X_train, y_train)
    pred = ensemble.predict(X_val)
    acc = accuracy_score(y_val, pred)
    f1 = f1_score(y_val, pred)
    print(f'  {name}: 验证准确率={acc:.4f}, F1={f1:.4f}')

# 选择表现最好的集成模型
ensemble_soft.fit(X, y)
final_model = ensemble_soft
final_model_name = 'Voting_Ensemble (Soft)'
print(f'\n  使用最终模型: {final_model_name}')

## 8️⃣ 生成提交文件

使用最佳模型对测试集进行预测，生成 Kaggle 提交格式的文件。

In [ ]:
# 生成提交文件
print('生成 Kaggle 提交文件 ...')

# 使用全量数据重新训练最终模型
final_model.fit(X, y)

# 预测测试集
predictions = final_model.predict(X_test)

# 创建提交文件
passenger_ids = test_df['PassengerId'].astype(int)
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions.astype(int),
})

# 保存
output_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'kaggle', 'titanic', 'submissions')
os.makedirs(output_dir, exist_ok=True)
submission_path = os.path.join(output_dir, 'titanic_submission.csv')
submission.to_csv(submission_path, index=False)

print(f'\n{"="*60}')
print('✅ 提交文件已生成！')
print(f'{"="*60}')
print(f'  文件路径: {submission_path}')
print(f'  样本数: {len(submission)}')
print(f'  生还预测比例: {submission["Survived"].mean():.2%}')
print(f'\n【提交文件预览】')
print(submission.head())

print(f'\n🚀 下一步:')
print(f'  1. 访问 https://www.kaggle.com/c/titanic/submit')
print(f'  2. 上传生成的 submission.csv')
print(f'  3. 查看排行榜: https://www.kaggle.com/c/titanic/leaderboard')
print(f'\n  祝好运！🚢✨')

In [ ]:
# 📝 最终总结
print('=' * 60)
print('📊 泰坦尼克号案例总结')
print('=' * 60)
print(f'\n🔧 使用的模型: {final_model_name}')
print(f'📈 最佳验证准确率: {results_df.iloc[0]["Val_Accuracy"]:.4f}')
print(f'📊 最佳CV准确率: {cv_df.iloc[0]["CV_Mean"]:.4f}')
print(f'🎯 使用的特征数: {X.shape[1]}')
print(f'📁 提交文件: {submission_path}')
print(f'\n📚 关键技术:')
print('  ✓ 缺失值处理 (中位数/众数填充)')
print('✓ 特征工程 (称谓/家庭规模/票价对数)')
print('  ✓ 类别特征编码 (One-Hot/Label)')
print('  ✓ 多模型对比 + 交叉验证')
print('  ✓ 超参数调优 (RandomizedSearchCV)')
print('  ✓ 集成学习 (Soft Voting)')
print(f'\n💡 改进方向:')
print('  1. 尝试 XGBoost / LightGBM')
print('  2. 特征交叉组合')
print('  3. Stacking 多层集成')
print('  4. 更精细的调参')